# Text Classification

Text classification is a natural language processing task that assigns labels or classes to text and has many practical applications.

Sentiment analysis is one of the most common text classification tasks, assigning labels such as 🙂positive and 🙁negative to text sequences.

In this material, you will learn:

1.  Use the [IMDb](https://huggingface.co/datasets/imdb) dataset and open-source [pretrained models](https://huggingface.co/models) to classify movie reviews as positive or negative.
2.  This material uses a DistillBert-based pretrained model: [jzonthemtn/distilbert-imdb](https://huggingface.co/jzonthemtn/distilbert-imdb)

## 0 Install libraries {#0-라이브러리-설치}

#### Using Hugging Face Transformers and Datasets {#hugging-face-transformersdatasets-활용}

In [ ]:
! pip install transformers datasets


#### Model inference

CPU / CUDA GPU: Automatically select an available device.

In [ ]:
import torch
pipeline_device = 0 if torch.cuda.is_available() else -1


## 1. Dataset {#1-dataset}

#### Load the IMDB Dataset (movie website reviews). {#imdb영화사이트-리뷰-dataset-가져오기}

Dataset [reference paper](https://aclanthology.org/P11-1015.pdf)

In [ ]:
from datasets import load_dataset

imdb = load_dataset('imdb')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

#### Print the first data item - data format: Dictionary ({key:value}) {#첫-번째-데이터-출력해보기---데이터-형태는-dictionarykeyvalue}

- text: list of review texts
- label: list of manually assigned target labels (0: negative, 1: positive)

In [ ]:
imdb['test'][0:3]

{'text': ['I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as

## 2 Inference using a pretrained model {#2-사전-학습-모델을-이용한-추론}

#### Load a pretrained model. {#사전-학습된-모델-불러오기}

In [ ]:
model_name = 'jzonthemtn/distilbert-imdb'
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

#### Try inference on a simple example. {#간단한-예시-추론해보기}

Using `pipeline()` is convenient.

`device = 0` is an argument supplied when using a GPU.

`sentiment-analysis` is an arbitrarily chosen pipeline name.

In [ ]:
from transformers import pipeline

classifier = pipeline('sentiment-analysis', model=model_name, device=pipeline_device)
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."
#text = "This was so bad!"
#text = "This was so good!"
classifier(text)

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

[{'label': 'LABEL_1', 'score': 0.9958142638206482}]

## 3 Evaluate the model {#3-모델-평가하기}

#### Accuracy

In [ ]:
import numpy as np

def classification_accuracy(predictions, labels):
    return {'accuracy': float(np.mean(np.asarray(predictions) == np.asarray(labels)))}

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return classification_accuracy(predictions, labels)


#### Randomly sample examples from the Test dataset and evaluate on them. {#test-dataset으로-부터-임의의-샘플을-random하게-추출하고-이를-이용해-평가한다}

In [ ]:
test_list = imdb['test']
test_sample_size = 20
rng = np.random.default_rng()
random_indices = rng.choice(len(test_list), size=test_sample_size, replace=False)
sampled_data = test_list.select(random_indices)

test_predictions = classifier(list(sampled_data['text']), max_length=512, truncation=True)
predicted_labels = [label['label'] for label in test_predictions]
mapped_labels = [model.config.label2id[label] for label in predicted_labels]
reference_labels = list(sampled_data['label'])
print(classification_accuracy(mapped_labels, reference_labels))
print(reference_labels)
print(mapped_labels)

{'accuracy': 1.0}
[1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0]
[1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0]


## 4 Assignment {#4-assignment}

#### a) Compare different pretrained models publicly available on Hugging Face. {#a-hugging-face에-공개된-다양한-사전학습-모델들을-바꿔가며-비교해본다}

#### b) Find a Korean sentence classification model using another Dataset, create a Test dataset of your own, and evaluate it. {#b-다른-dataset을-이용한-한글-문장-분류-모델을-찾아보고-임의로-test-dataset을-만들어-평가해본다}

In [ ]:
ko_sentiment_data = load_dataset('sepidmnorozy/Korean_sentiment')

train.csv:   0%|          | 0.00/3.29M [00:00<?, ?B/s]

dev.csv:   0%|          | 0.00/127k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/239k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1333 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2667 [00:00<?, ? examples/s]

In [ ]:
ko_sentiment_data['test'][0:3]

{'label': [0, 1, 1],
 'text': ['이 영화는 아름다운 영상미로 쓰레기같은 내용을 감추고 멋진 영화로 위장을 하고있다. 나는 이 영화가 만들다가 만 영화라고 생각한다.',
  '워메 ㅋㅋ 아이언맨이 여기나오는줄은 ㅋㅋ반전이네요',
  '여자 진짜 개귀엽다 ㅋㅋㅋㅋ']}

In [ ]:
model_name = 'WhitePeak/bert-base-cased-Korean-sentiment'
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(model_name)
#classifier = pipeline('sentiment-analysis', model=model_name, device=pipeline_device)
classifier = pipeline('text-classification', model=model_name, device=pipeline_device)
classifier(ko_sentiment_data['test'][3]['text'])

[{'label': 'LABEL_1', 'score': 0.9814258813858032}]

In [ ]:
test_list = ko_sentiment_data['test']
test_sample_size = 100
rng = np.random.default_rng()
random_indices = rng.choice(len(test_list), size=test_sample_size, replace=False)
sampled_data = test_list.select(random_indices)

test_predictions = classifier(list(sampled_data['text']), max_length=512, truncation=True)
predicted_labels = [label['label'] for label in test_predictions]
mapped_labels = [model.config.label2id[label] for label in predicted_labels]
reference_labels = list(sampled_data['label'])
print(classification_accuracy(mapped_labels, reference_labels))
print(reference_labels)
print(mapped_labels)

{'accuracy': 0.77}
[0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
